# Object Detection with YOLOS-Small

Este notebook demonstra como usar o modelo `hustvl/yolos-small` do Hugging Face para realizar detecção de objetos em imagens locais.

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import YolosImageProcessor, YolosForObjectDetection

# Verificar se há GPU disponível
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando o dispositivo: {device}")

In [ ]:
# Carregar o processador e o modelo
model_name = "hustvl/yolos-small"
processor = YolosImageProcessor.from_pretrained(model_name)
model = YolosForObjectDetection.from_pretrained(model_name).to(device)

print("Modelo carregado com sucesso!")

In [ ]:
def detect_objects(image_path, threshold=0.9):
    """
    Realiza a detecção de objetos em uma imagem e exibe o resultado.
    """
    image = Image.open(image_path).convert("RGB")
    
    # Preparar a imagem para o modelo
    inputs = processor(images=image, return_tensors="pt").to(device)
    
    # Fazer a predição
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Pós-processamento para converter os resultados em caixas delimitadoras e rótulos
    target_sizes = torch.tensor([image.size[::-1]])
    results = processor.post_process_object_detection(outputs, threshold=threshold, target_sizes=target_sizes)[0]
    
    # Visualização
    plt.figure(figsize=(12, 8))
    plt.imshow(image)
    ax = plt.gca()
    
    colors = plt.cm.hsv(torch.linspace(0, 1, 21)).tolist() # Cores aleatórias para as caixas
    
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        box = [round(i, 2) for i in box.tolist()]
        label_name = model.config.id2label[label.item()]
        
        # Adicionar retângulo
        rect = plt.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                            fill=False, color='red', linewidth=2)
        ax.add_patch(rect)
        
        # Adicionar rótulo com pontuação
        text = f'{label_name}: {score.item():.2f}'
        ax.text(box[0], box[1], text, fontsize=12, 
                bbox=dict(facecolor='yellow', alpha=0.5))
    
    plt.axis('off')
    plt.title(f"Detecções em {os.path.basename(image_path)}")
    plt.show()

In [ ]:
# Listar e processar todas as imagens na pasta 'imagens'
image_dir = "imagens"
image_files = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

if not image_files:
    print(f"Nenhuma imagem encontrada na pasta {image_dir}")
else:
    for image_path in image_files:
        print(f"Processando {image_path}...")
        detect_objects(image_path)